# Measurement DB assets

Read the asset types and the stored `assets.parquet` size for **every** benchmark that publishes assets. The notebook pins one repository revision and never reads asset payloads.

In [18]:
import json
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import httpx
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, HfFileSystem
from huggingface_hub.hf_api import RepoFolder
from IPython.display import display

REPO = "aims-foundations/measurement-db"
API = HfApi()


def retry_network(operation, label, attempts=5):
    """Retry transient Hub disconnects; data and programming errors fail immediately."""
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except (httpx.HTTPError, OSError, TimeoutError) as error:
            if attempt == attempts:
                raise
            delay = min(2 ** (attempt - 1), 8)
            print(f"{label}: {type(error).__name__}; retry {attempt}/{attempts} in {delay}s")
            time.sleep(delay)


REVISION = retry_network(
    lambda: API.dataset_info(REPO).sha, "Read repository revision",
)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)
print("Measurement DB revision:", REVISION)

Measurement DB revision: 54074453da7b9911be0f48e4c560be547bc84d21


## 1. Discover every asset-bearing benchmark

In [19]:
entries = retry_network(
    lambda: list(API.list_repo_tree(
        REPO, repo_type="dataset", revision=REVISION, recursive=False,
    )),
    "List benchmark directories",
)
benchmarks = sorted(entry.path for entry in entries if isinstance(entry, RepoFolder))
candidate_paths = [f"{benchmark}/formatted_tables/assets.parquet" for benchmark in benchmarks]
asset_file_info = retry_network(
    lambda: API.get_paths_info(
        REPO, candidate_paths, repo_type="dataset", revision=REVISION, expand=True,
    ),
    "Find asset tables",
)
asset_parquet_bytes = {info.rfilename.split("/", 1)[0]: info.size for info in asset_file_info}
asset_benchmarks = sorted(asset_parquet_bytes)
print(f"{len(asset_benchmarks)} of {len(benchmarks)} benchmarks have assets.parquet")
asset_benchmarks

66 of 281 benchmarks have assets.parquet


['agc_bench',
 'ai2d_test',
 'alpha_sql',
 'auditing_sabotage_bench',
 'averimatec',
 'babilong',
 'brace',
 'chartmuseum',
 'chipbench',
 'dataclawbench',
 'dave',
 'devbench',
 'disco',
 'edu_circuit_hw',
 'ehrflowbench',
 'engdesign',
 'enginemt_qa',
 'exploitgym',
 'financebench',
 'genai_learning',
 'goodai_ltm_benchmark',
 'haiid',
 'hallusionbench',
 'harmbench',
 'hle',
 'igakuqa119',
 'image2struct',
 'imagenet_hard',
 'imgedit',
 'intercode',
 'kris_bench',
 'legal_rag_bench',
 'live_agent_risk',
 'llmail_inject',
 'matharena',
 'mathvista_mini',
 'medagentboard',
 'mj_bench',
 'mmbench_v11',
 'mmmu_dev_val',
 'mmoral_bench',
 'mochi',
 'mtbbench',
 'multimodal_stem_ai',
 'naep_llm_students',
 'nis3d',
 'nyu_ctf_bench',
 'ocrbench_v2',
 'phyblock',
 'qatch',
 'rclicks',
 'researchclawbench',
 'risebench',
 'scbench_long',
 'sketchjudge',
 'stanford_orb',
 'sugarcrepe',
 'threeeed',
 'vhelm',
 'visit_bench',
 'visual_memory',
 'visual_riddles',
 'visualwebarena',
 'vl_rewardbe

## 2. Read every type and benchmark size

Types come from `items.parquet:asset_manifest`. Benchmark size is the stored byte size of its complete `assets.parquet`, obtained from Hugging Face file metadata during discovery. The asset Parquet files themselves are not downloaded or opened.

In [20]:
def remote_columns(benchmark, table, columns):
    uri = (
        f"hf://datasets/{REPO}@{REVISION}/"
        f"{benchmark}/formatted_tables/{table}.parquet"
    )

    def read():
        # A fresh filesystem object gives every retry a fresh connection.
        with HfFileSystem().open(uri, "rb") as file:
            return pq.read_table(file, columns=columns).to_pydict()

    return retry_network(read, f"Read {benchmark}/{table}")


def scan_benchmark(benchmark):
    manifests = remote_columns(benchmark, "items", ["asset_manifest"])["asset_manifest"]
    asset_ids_by_type = defaultdict(set)
    reference_counts = Counter()
    item_counts_by_type = Counter()
    items_with_assets = 0
    for raw_manifest in manifests:
        if not raw_manifest:
            continue
        assets = json.loads(raw_manifest)
        if not assets:
            continue
        items_with_assets += 1
        media_types_for_item = set()
        for asset in assets:
            media_type = asset.get("media_type") or "<unknown>"
            asset_ids_by_type[media_type].add(asset["asset_id"])
            reference_counts[media_type] += 1
            media_types_for_item.add(media_type)
        item_counts_by_type.update(media_types_for_item)

    all_asset_ids = set().union(*asset_ids_by_type.values()) if asset_ids_by_type else set()
    benchmark_row = {
        "benchmark": benchmark,
        "media_types": ";".join(sorted(asset_ids_by_type)),
        "asset_ids_in_manifests": len(all_asset_ids),
        "manifest_references": sum(reference_counts.values()),
        "items_with_assets": items_with_assets,
        "assets_parquet_bytes": asset_parquet_bytes[benchmark],
    }
    type_rows = [{
        "benchmark": benchmark,
        "media_type": media_type,
        "asset_ids": len(asset_ids),
        "manifest_references": reference_counts[media_type],
        "items_with_media_type": item_counts_by_type[media_type],
    } for media_type, asset_ids in asset_ids_by_type.items()]
    return benchmark_row, type_rows


with ThreadPoolExecutor(max_workers=6) as pool:
    scans = list(pool.map(scan_benchmark, asset_benchmarks))

print(f"Read types and stored sizes for all {len(scans)} asset-bearing benchmarks")

Read types and stored sizes for all 66 asset-bearing benchmarks


In [21]:
def human_bytes(value):
    value = float(value)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if value < 1024 or unit == "TiB":
            return f"{value:,.1f} {unit}"
        value /= 1024


type_rows = pd.DataFrame([row for _, rows in scans for row in rows])
type_summary = (type_rows.groupby("media_type", as_index=False)
                .agg(benchmarks=("benchmark", "nunique"),
                     asset_ids=("asset_ids", "sum"),
                     manifest_references=("manifest_references", "sum"))
                .sort_values("media_type"))
print(f"{len(type_summary)} media types")
display(type_summary)

benchmark_sizes = pd.DataFrame([row for row, _ in scans])
benchmark_sizes["assets_parquet_size"] = benchmark_sizes.assets_parquet_bytes.map(human_bytes)
benchmark_sizes = benchmark_sizes.sort_values("assets_parquet_bytes", ascending=False)

display(benchmark_sizes[[
    "benchmark", "media_types", "items_with_assets", "asset_ids_in_manifests",
    "manifest_references", "assets_parquet_bytes", "assets_parquet_size",
]])

26 media types


,media_type,benchmarks,asset_ids,manifest_references
0,application/gzip,2,13,24
1,application/json,8,4484,213271
2,application/octet-stream,6,17645,40577
3,application/pdf,5,267,428
4,application/sql,1,1,2068
5,application/vnd.apache.parquet,1,2,20
6,application/vnd.sqlite3,2,170,22744
7,application/x-ndjson,1,1,300
8,application/x-npy,2,1282,1930
9,application/zip,2,21,39


,benchmark,media_types,items_with_assets,asset_ids_in_manifests,manifest_references,assets_parquet_bytes,assets_parquet_size
55,stanford_orb,application/x-npy;image/png;image/x-exr;model/obj,630,10221,184394,93768574753,87.3 GiB
12,disco,image/png,14000,13999,14000,9638010661,9.0 GiB
58,vhelm,image/jpeg;image/png,93127,26205,93267,8798372026,8.2 GiB
10,dave,audio/mpeg;video/mp4,3922,4193,7844,5025159889,4.7 GiB
37,mj_bench,image/jpeg;image/png;image/webp,26564,5241,43536,4133259893,3.8 GiB
60,visual_memory,image/jpeg;image/png,47500,47437,47500,4103727681,3.8 GiB
6,brace,audio/wav,12631,2027,12631,3765783271,3.5 GiB
65,wonderbread,image/png,8061,4954,187217,3724357501,3.5 GiB
5,babilong,text/plain,525502,49045,525502,2910142091,2.7 GiB
57,threeeed,application/octet-stream;image/jpeg,8616,8914,17232,2462367842,2.3 GiB


In [22]:
# An item is counted once per media type, regardless of how many assets of that type it references.
# Consequently, an item with more than one media type appears in more than one row.
total_items_with_assets = benchmark_sizes.items_with_assets.sum()
item_media_type_summary = (
    type_rows.groupby("media_type", as_index=False)
    .agg(
        items=("items_with_media_type", "sum"),
        benchmarks=("benchmark", "nunique"),
    )
    .assign(share_of_asset_items=lambda frame: frame["items"] / total_items_with_assets)
    .sort_values("items", ascending=False)
)
print(f"{total_items_with_assets:,} items have at least one asset")
display(item_media_type_summary.style.format({"share_of_asset_items": "{:.1%}"}))

1,136,022 items have at least one asset


,media_type,items,benchmarks,share_of_asset_items
22,text/plain,541275,7,47.6%
14,image/jpeg,254548,30,22.4%
1,application/json,208086,8,18.3%
15,image/png,114769,34,10.1%
6,application/vnd.sqlite3,22744,2,2.0%
11,audio/wav,12631,1,1.1%
2,application/octet-stream,10863,6,1.0%
25,video/mp4,3922,1,0.3%
10,audio/mpeg,3922,1,0.3%
20,text/csv,2192,4,0.2%
